# H4. Hardware sessions on ibm_fez

Runs the pre-registered schedule of notebook H3: **session 0** (a dry run of about 40 seconds that measures the cost
with gate twirling on), then **sessions 1 to 8, one per calibration day**. Each session submits 2 jobs. Every job carries
rows from all three arms (R, C, N), so all arms see the same hardware conditions.

| Setting | Value | Why |
|---|---|---|
| Device | ibm_fez (Heron r2) | HSBC reproduced their result on fez and marrakesh |
| Circuit | "longer" PQFM on a 129-qubit chain (HSBC: 109 qubits on ibm_torino) | HSBC's main circuit, at more qubits |
| Arms | R (random), C (chronological), N (null circuit); the same 1,000 events | see H3 |
| Shots | 1,024 per basis, every arm | buys 1,000 events per arm, the most the budget allows |
| Mitigation | Pauli twirling on gates and readout + TREX, no ZNE | as HSBC |
| Sessions | 8, each on a new calibration | HSBC's runs spanned many days; batch fingerprints live there |
| Budget | never passes 50 min; the null arm is halved if session 0 shows the plan would pass it | pre-registered rule |

**How to run**
1. Run All with `SUBMIT = False` (as delivered). It checks the account, the pre-registration fingerprint, the chain
   and the cost, then stops.
2. Set `SUBMIT = True` and Run All: submits **session 0**. When it has finished (minutes to hours, depending on the
   queue), Run All again: it collects session 0, prints the measured cost and applies the budget rule.
3. From then on, **Run All once per calibration day** (see SCHEDULE.md). Each run collects what has finished and
   submits the next session only if the previous one is collected and the device has been recalibrated. It never
   resubmits a job.

To check the code without IBM, set `USE_HARDWARE = False`: the circuit is transpiled on FakeFez and a 6-qubit
miniature of session 0 runs on the local simulator.

In [ ]:
USE_HARDWARE      = True
SUBMIT            = False
MAX_TOTAL_MINUTES = 50.5
ACCOUNT, DEVICE   = "Faryad-free-ibm", "ibm_fez"
SHOTS             = 1024
FORCE_SAME_CALIBRATION = False   # True allows two sessions on one calibration. That breaks the design; avoid.

In [ ]:
import os, sys, json, time, math, warnings
from pathlib import Path
# Find the study folder (the one holding the hsbcrep package), wherever Jupyter or VS Code started
def _find_root():
    starts = [Path(p).parent for p in [globals().get("__vsc_ipynb_file__")] if p]
    starts += [Path.cwd()]
    for s in starts:
        for p in [s, *s.parents]:
            if (p / "hsbcrep" / "__init__.py").exists():
                return p.resolve()
    raise RuntimeError(f"Cannot find the repository folder (the one holding hsbcrep/) from {Path.cwd()}. Set ROOT by hand in this cell.")
ROOT = _find_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repository folder:", ROOT)
OUT = ROOT / "outputs"; OUT.mkdir(exist_ok=True)
RAW = ROOT / "data" / "raw"
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
pd.set_option("display.width", 200)
import matplotlib.pyplot as plt
from hsbcrep import pqfm, hardware as LH, design as D
from hsbcrep.backends import local_backend

class Stop(Exception):
    def _render_traceback_(self):
        return [f"STOPPED: {self}"]

HWD = OUT / "hardware"; HWD.mkdir(exist_ok=True)
pre = OUT / "prereg"
S = pd.read_csv(pre / "schedule.csv")
plan = json.loads((pre / "plan.json").read_text())
J = np.load(OUT / "J_all.npy")

## H4.1 Pre-registration check

Recomputes the fingerprint of `outputs/prereg/` and of the analysis code. It must match the one H3 printed.

In [ ]:
frozen = (pre / "SHA256.txt").read_text().strip()
now = D.prereg_digest(ROOT)
print("frozen :", frozen); print("now    :", now)
if now != frozen:
    raise Stop("the pre-registered files or the code changed since H3 froze them. Restore them, or (before any session) re-freeze in H3.")

## H4.2 Backend, chain and circuit

The 129-qubit chain (`outputs/chains129.json`, the longest simple path found on ibm_fez's heavy-hex lattice) is fixed
at session 0 and saved; every later session reuses it.

In [ ]:
if USE_HARDWARE:
    from qiskit_ibm_runtime import QiskitRuntimeService
    if ACCOUNT not in QiskitRuntimeService.saved_accounts():
        raise Stop(f"no saved account named {ACCOUNT!r}")
    service = QiskitRuntimeService(name=ACCOUNT)
    backend = service.backend(DEVICE)
    print(f"{backend.name}: operational={backend.status().operational}, pending jobs={backend.status().pending_jobs}")
    try:
        print("account usage:", service.usage())
    except Exception as e:
        print("usage not available:", type(e).__name__)
    N129 = 129
else:
    service, backend = None, local_backend()
    N129 = 6                                                   # miniature for the local check

chain_file = HWD / ("chain.json" if USE_HARDWARE else "chain_local.json")
if chain_file.exists():
    chain = json.loads(chain_file.read_text())["chain"]
elif USE_HARDWARE:
    chain, sc, k = LH.choose_chain(backend, json.loads((OUT / "chains129.json").read_text()))
    LH.save_json(chain_file, {"chain": chain, "score": sc, "chosen_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
                              "calibration": LH.calibration_stamp(backend)})
else:
    chain = pqfm.find_chain(backend.coupling_map.get_edges(), N129, starts=[0])
    LH.save_json(chain_file, {"chain": chain})
isa, cols, obs = LH.build_isa(N129, pqfm.pqfm(N129, 216, "longer"), 216, chain, backend)
cz, depth = LH.two_q_count(isa)
print(f"{len(chain)}-qubit chain: {cz} two-qubit gates, two-qubit depth {depth}, {isa.estimate_duration(backend.target) * 1e6:.1f} us per circuit")

## H4.3 Bookkeeping and collection

`outputs/hardware/log.json` keeps every job: its session, the calibration stamp at submission, the rows it carried,
and later the billed time. This cell also collects every finished job.

In [ ]:
LOG = HWD / ("log.json" if USE_HARDWARE else "log_local.json")
log = LH.load_json(LOG, {"sessions": {}})

def rows_for(session, job):
    r = S[(S.session == session) & (S.job == job)].reset_index(drop=True)
    if log.get("halve_null"):                                  # pre-registered budget rule
        keep = ~r.null.values | (np.cumsum(r.null.values) % 2 == 1)
        r = r[keep].reset_index(drop=True)
    return r

def values(rows):
    V = J[rows.event.values][:, cols].copy()
    V[rows.null.values.astype(bool)] = 0.0                     # null arm: same circuit, zero data
    return V

In [ ]:
LOCAL = globals().get("LOCAL", {})
def collect():
    changed = False
    for skey, sess in log["sessions"].items():
        for j in sess["jobs"]:
            if j.get("collected"):
                continue
            try:
                job = service.job(j["id"]) if USE_HARDWARE else LOCAL[j["id"]]
            except KeyError:
                print(f"session {skey}: local job from an earlier kernel; delete {LOG.name} to redo the local check"); continue
            st = str(job.status())
            if st in ("ERROR", "CANCELLED"):
                print(f"session {skey} job {j['job']}: {st}. Check the IBM dashboard."); continue
            if st != "DONE":
                print(f"session {skey} job {j['job']}: {st}"); continue
            (evs, std), = LH.unpack(job.result(), [N129])
            np.save(HWD / f"s{skey}_j{j['job']}_evs.npy", evs); np.save(HWD / f"s{skey}_j{j['job']}_std.npy", std)
            try:
                j["usage_s"] = float(job.usage())
            except Exception:
                j["usage_s"] = None
            try:                                            # when the job actually ran (for the fingerprint analysis)
                m = job.metrics()
                j["timestamps"] = m.get("timestamps", {}); j["usage"] = m.get("usage", {})
            except Exception:
                pass
            j["collected"] = True; changed = True
            print(f"session {skey} job {j['job']}: collected ({j['usage_s']} s billed)")
    if changed:
        LH.save_json(LOG, log)
collect()

## H4.4 Cost and the budget rule

The cost per shot is refitted on every job IBM has billed. After session 0 is collected, and before session 1 is
submitted, the pre-registered budget rule is applied once: if the projected total exceeds the cap, the null arm is
halved for all sessions.

In [ ]:
jobs_done = [j for s in log["sessions"].values() for j in s["jobs"] if j.get("usage_s")]
billed = sum(j["usage_s"] for j in jobs_done)
us_shot, ovh = 284.0, 11.4                                     # PQK100 pilot on ibm_fez, no gate twirling
if len(jobs_done) >= 2:
    A = np.c_[np.ones(len(jobs_done)), [len(j["events"]) * 3 * SHOTS for j in jobs_done]]
    (ovh, b1), *_ = np.linalg.lstsq(A, np.array([j["usage_s"] for j in jobs_done]), rcond=None)
    us_shot = b1 * 1e6
    print(f"rate refitted on {len(jobs_done)} billed jobs: {us_shot:.0f} us per shot + {ovh:.1f} s per job")
elif len(jobs_done) == 1:
    print("only one billed job so far; using the pilot rate until session 0 is fully collected")

def session_minutes(s):
    n = sum(len(rows_for(s, j)) for j in range(plan["jobs_per_session"]))
    return (n * 3 * SHOTS * us_shot * 1e-6 + plan["jobs_per_session"] * ovh) / 60

remaining = [s for s in range(plan["sessions"]) if str(s + 1) not in log["sessions"]]
proj = billed / 60 + sum(session_minutes(s) for s in remaining)
print(f"billed so far {billed / 60:.1f} min; projected total {proj:.1f} min (cap {MAX_TOTAL_MINUTES})")

if ("0" in log["sessions"] and all(j.get("collected") for j in log["sessions"]["0"]["jobs"])
        and "1" not in log["sessions"] and "halve_null" not in log):
    log["halve_null"] = bool(proj > MAX_TOTAL_MINUTES)
    LH.save_json(LOG, log)
    print("budget rule applied after session 0: null arm", "HALVED" if log["halve_null"] else "kept in full")
    if log["halve_null"]:
        proj = billed / 60 + sum(session_minutes(s) for s in remaining)
        print(f"new projected total {proj:.1f} min")

## H4.5 Submit the next session

Session 0 is the dry run: two jobs of 4 and 20 rows taken from schedule session 0 (not scored). Sessions 1 to 8 run schedule sessions 0
to 7 in full. A session is submitted only if (a) the previous one is collected, (b) the device has been recalibrated
since the previous one, and (c) the projected total stays under the cap.

In [ ]:
def submit_session(skey):
    s_sched = 0 if skey == "0" else int(skey) - 1
    est = LH.make_estimator(backend, SHOTS, hardware=USE_HARDWARE)
    sess = {"submitted_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
            "calibration": LH.calibration_stamp(backend) if USE_HARDWARE else "", "shots": SHOTS,
            "schedule_session": s_sched, "jobs": []}
    if USE_HARDWARE:
        try:
            LH.save_json(HWD / f"calibration_session{skey}.json", backend.properties().to_dict())
        except Exception as e:
            print("calibration not saved:", type(e).__name__)
    if skey == "0":                                            # dry run: two jobs of 4 and 20 rows, so the cost fit
        r0 = rows_for(0, 0)                                    # separates the per-job overhead from the per-shot cost
        parts = [(0, r0.iloc[:4]), (1, r0.iloc[4:24])]
    else:
        parts = [(jj, rows_for(s_sched, jj)) for jj in range(plan["jobs_per_session"])]
    for jj, rows in parts:
        rows = rows.reset_index(drop=True)
        job = est.run([(isa, obs, values(rows))])
        if not USE_HARDWARE:
            LOCAL[job.job_id()] = job
        sess["jobs"].append({"job": jj, "id": job.job_id(), "events": rows.event.tolist(), "arms": rows.arm.tolist()})
        log["sessions"][skey] = sess
        LH.save_json(LOG, log)                                  # saved after every job
    print(f"session {skey}: submitted {len(sess['jobs'])} job(s)")

def cz_dates(props):
    """Calibration date of the CZ gate on every bond of the chain (added Oct 22, 2026)."""
    bonds = {tuple(sorted(p)) for p in zip(chain[:-1], chain[1:])}
    out = {}
    for g in props["gates"]:
        if g["gate"] == "cz" and tuple(sorted(g["qubits"])) in bonds:
            for p in g["parameters"]:
                if p["name"] == "gate_error":
                    out[tuple(sorted(g["qubits"]))] = str(p["date"])
    return out

done = sorted(log["sessions"], key=int)
nxt = "0" if not done else str(int(done[-1]) + 1)
if int(nxt) > plan["sessions"]:
    print("all sessions submitted.")
elif not USE_HARDWARE and int(nxt) > 0:
    print("local check complete (session 0 only).")
else:
    prev = log["sessions"].get(str(int(nxt) - 1)) if nxt != "0" else None
    blockers = []
    if prev and not all(j.get("collected") for j in prev["jobs"]):
        blockers.append("the previous session is not collected yet; run again later")
    if USE_HARDWARE and prev and prev["calibration"] and prev["calibration"] == LH.calibration_stamp(backend) and not FORCE_SAME_CALIBRATION:
        blockers.append(f"no recalibration since the previous session ({prev['calibration']}); try again after the next calibration")
    if USE_HARDWARE and prev and not FORCE_SAME_CALIBRATION:     # a readout-only update is not a new calibration
        f_prev = HWD / f"calibration_session{int(nxt) - 1}.json"
        if f_prev.exists():
            try:
                props_now = backend.properties(refresh=True)
            except TypeError:
                props_now = backend.properties()
            old = cz_dates(json.loads(f_prev.read_text()))
            new = cz_dates(json.loads(json.dumps(props_now.to_dict(), default=str)))
            frac = float(np.mean([old.get(b) != new.get(b) for b in new])) if new else 0.0
            print(f"two-qubit gates on the chain recalibrated since the previous session: {frac:.0%}")
            if frac < 0.5:
                blockers.append(f"only {frac:.0%} of the chain's CZ gates were recalibrated since the previous session "
                                "(a readout-only update); wait for the next full calibration, usually the next day")
    if nxt != "0" and billed / 60 + session_minutes(int(nxt) - 1) > MAX_TOTAL_MINUTES:
        blockers.append("this session would pass MAX_TOTAL_MINUTES")
    if blockers:
        print(f"session {nxt} not submitted: " + "; ".join(blockers))
    elif not SUBMIT:
        print(f"session {nxt} is due. Set SUBMIT = True and Run All to submit it.")
    else:
        submit_session(nxt)
        if not USE_HARDWARE:
            collect()

## H4.6 Status

In [ ]:
st = [{"session": k, "submitted (UTC)": v["submitted_utc"], "calibration": v["calibration"], "jobs": len(v["jobs"]),
       "collected": sum(bool(j.get("collected")) for j in v["jobs"]),
       "billed s": round(sum(j.get("usage_s") or 0 for j in v["jobs"]), 1)} for k, v in sorted(log["sessions"].items(), key=lambda kv: int(kv[0]))]
print(f"halve null arm: {log.get('halve_null', 'not decided yet')}")
pd.DataFrame(st) if st else print("no session submitted yet")